# Tutorial 4: Working with JANIS (JAva-based Nuclear Data Information System)

## Overview

JANIS is a web-based system maintained by the OECD NEA that provides unified access to multiple nuclear data libraries. Instead of downloading files from different sources, you can query multiple databases through one interface.

### What you'll learn:
- How to access JANIS web interface
- How to query multiple libraries simultaneously
- How to use web scraping/API calls to retrieve data
- How to compare data from different sources (ENDF, JEFF, JENDL, etc.)
- Creating automated data retrieval workflows

### Prerequisites:
```bash
pip install requests beautifulsoup4 matplotlib numpy pandas lxml
```

## 1. Understanding JANIS

### Key Features:

- **Multi-library Access**: Query ENDF, JEFF, JENDL, TENDL, and more
- **Web Interface**: https://www.oecd-nea.org/janisweb/
- **Interactive Plots**: View cross-sections in browser
- **Data Export**: Download data in various formats
- **Comparison Tools**: Side-by-side library comparison

### Available Libraries in JANIS:

- **ENDF/B-VIII.0** (USA)
- **JEFF-3.3** (Europe)
- **JENDL-5.0** (Japan)
- **TENDL-2021** (PSI/IAEA)
- **CENDL** (China)
- **ROSFOND** (Russia)
- **EXFOR** (Experimental)

In [ ]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import json
import time

# Create data directory
data_dir = Path('../data/janis')
data_dir.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

## 2. Accessing JANIS Programmatically

While JANIS has a web interface, we can also access it programmatically. Note that JANIS doesn't have an official REST API, so we'll demonstrate the concept with simulated data.

### JANIS Web Interface:
- URL: https://www.oecd-nea.org/janisweb/
- Select isotope, reaction, and libraries to compare
- View interactive plots
- Export data as CSV or text

In [ ]:
def janis_web_query_info():
    """
    Information about using JANIS web interface.
    """
    info = {
        'url': 'https://www.oecd-nea.org/janisweb/',
        'steps': [
            '1. Go to JANIS web interface',
            '2. Select "Nuclear Data" → "Cross Sections"',
            '3. Choose isotope (e.g., U-235)',
            '4. Select reaction (e.g., (n,f) fission)',
            '5. Choose libraries to compare (ENDF, JEFF, JENDL, etc.)',
            '6. View interactive plot',
            '7. Export data using "Export" button'
        ],
        'export_formats': ['CSV', 'Text', 'ENDF'],
        'available_libraries': [
            'ENDF/B-VIII.0',
            'JEFF-3.3',
            'JENDL-5.0',
            'TENDL-2021',
            'CENDL-3.2',
            'ROSFOND-2010'
        ]
    }
    return info

info = janis_web_query_info()
print("JANIS Web Interface Information:")
print(f"\nURL: {info['url']}")
print("\nSteps to use JANIS:")
for step in info['steps']:
    print(f"  {step}")
print(f"\nAvailable libraries: {', '.join(info['available_libraries'])}")

## 3. Simulating Multi-Library Comparison

Let's simulate what data from different libraries looks like for the same reaction (U-235 fission).

In [ ]:
def simulate_library_data(library_name, isotope='U-235', reaction='fission'):
    """
    Simulate cross-section data from different libraries.
    In practice, you would download this from JANIS.
    """
    energy = np.logspace(-2, 7, 5000)
    
    # Base cross-section
    thermal = 584.0 * np.sqrt(0.0253 / np.maximum(energy, 1e-5))
    resonance = 50 + 30 * np.sin(np.log(np.maximum(energy, 1)))
    fast = 2.0 + 0.5 * np.log10(np.maximum(energy, 1e5) / 1e5)
    xs_base = np.where(energy < 1, thermal, np.where(energy < 1e4, resonance, fast))
    
    # Add library-specific variations
    if library_name == 'ENDF/B-VIII.0':
        xs = xs_base * 1.000  # Reference
    elif library_name == 'JEFF-3.3':
        xs = xs_base * 0.995  # Slightly lower
    elif library_name == 'JENDL-5.0':
        xs = xs_base * 1.005  # Slightly higher
    elif library_name == 'TENDL-2021':
        xs = xs_base * 0.990  # Lower in some regions
    elif library_name == 'CENDL-3.2':
        xs = xs_base * 1.002
    else:
        xs = xs_base
    
    return energy, xs

# Generate data from multiple libraries
libraries = ['ENDF/B-VIII.0', 'JEFF-3.3', 'JENDL-5.0', 'TENDL-2021', 'CENDL-3.2']
library_data = {}

for lib in libraries:
    energy, xs = simulate_library_data(lib)
    library_data[lib] = {'energy': energy, 'xs': xs}

print(f"Generated data for {len(libraries)} libraries")
for lib in libraries:
    print(f"  - {lib}: {len(library_data[lib]['energy'])} data points")

## 4. Multi-Library Comparison Plot

Let's create a comparison plot showing all libraries on the same axes.

In [ ]:
plt.figure(figsize=(14, 8))

colors = ['blue', 'red', 'green', 'orange', 'purple']
linestyles = ['-', '--', '-.', ':', '-']

for i, lib in enumerate(libraries):
    data = library_data[lib]
    plt.loglog(data['energy'], data['xs'], 
               label=lib, 
               color=colors[i], 
               linestyle=linestyles[i],
               linewidth=2,
               alpha=0.8)

plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('Fission Cross-section (barns)', fontsize=14)
plt.title('U-235 Fission Cross-Section: Multi-Library Comparison (JANIS)', fontsize=16)
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3, which='both')
plt.xlim(1e-2, 1e7)
plt.tight_layout()
plt.savefig(data_dir / 'janis_multi_library_comparison.png', dpi=150)
plt.show()

## 5. Detailed Ratio Analysis

Let's examine the ratios of different libraries relative to ENDF (the reference).

In [ ]:
# Use ENDF as reference
reference = library_data['ENDF/B-VIII.0']

fig, axes = plt.subplots(2, 1, figsize=(14, 10), 
                         gridspec_kw={'height_ratios': [3, 2]})

# Top panel: Cross-sections
for i, lib in enumerate(libraries):
    data = library_data[lib]
    axes[0].loglog(data['energy'], data['xs'], 
                   label=lib, 
                   color=colors[i], 
                   linestyle=linestyles[i],
                   linewidth=2,
                   alpha=0.8)

axes[0].set_ylabel('Fission Cross-section (barns)', fontsize=14)
axes[0].set_title('U-235 Fission: All Libraries with Ratios to ENDF', fontsize=16)
axes[0].legend(fontsize=10, loc='best')
axes[0].grid(True, alpha=0.3, which='both')
axes[0].set_xlim(1e-2, 1e7)

# Bottom panel: Ratios to ENDF
for i, lib in enumerate(libraries[1:], 1):  # Skip ENDF itself
    data = library_data[lib]
    ratio = data['xs'] / reference['xs']
    axes[1].semilogx(data['energy'], ratio, 
                     label=f'{lib}/ENDF', 
                     color=colors[i], 
                     linewidth=2)

axes[1].axhline(1.0, color='black', linestyle='-', linewidth=1.5, alpha=0.7)
axes[1].axhline(1.02, color='gray', linestyle='--', linewidth=1, alpha=0.5)
axes[1].axhline(0.98, color='gray', linestyle='--', linewidth=1, alpha=0.5)
axes[1].set_xlabel('Neutron Energy (eV)', fontsize=14)
axes[1].set_ylabel('Ratio to ENDF/B-VIII.0', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(1e-2, 1e7)
axes[1].set_ylim(0.98, 1.02)

plt.tight_layout()
plt.savefig(data_dir / 'janis_ratio_analysis.png', dpi=150)
plt.show()

## 6. Energy Region Analysis

Let's examine how libraries differ in specific energy regions.

In [ ]:
# Define energy regions
regions = {
    'Thermal': (0.01, 1.0),
    'Epithermal': (1.0, 100.0),
    'Resonance': (100.0, 10000.0),
    'Fast': (10000.0, 1e7)
}

# Calculate statistics for each region
region_stats = []

for region_name, (e_min, e_max) in regions.items():
    for lib in libraries[1:]:
        # Get data in this energy range
        data = library_data[lib]
        ref_data = library_data['ENDF/B-VIII.0']
        
        mask = (data['energy'] >= e_min) & (data['energy'] <= e_max)
        
        if np.any(mask):
            ratio = data['xs'][mask] / ref_data['xs'][mask]
            region_stats.append({
                'Region': region_name,
                'Library': lib,
                'Mean_Ratio': np.mean(ratio),
                'Std_Ratio': np.std(ratio),
                'Min_Ratio': np.min(ratio),
                'Max_Ratio': np.max(ratio)
            })

df_regions = pd.DataFrame(region_stats)

# Pivot for easier viewing
df_pivot = df_regions.pivot(index='Library', columns='Region', values='Mean_Ratio')

print("Mean Ratio to ENDF by Energy Region:")
print(df_pivot)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
df_pivot.plot(kind='bar', ax=ax, width=0.8)
ax.axhline(1.0, color='black', linestyle='--', linewidth=2)
ax.set_ylabel('Mean Ratio to ENDF/B-VIII.0', fontsize=12)
ax.set_xlabel('Library', fontsize=12)
ax.set_title('Library Comparison by Energy Region', fontsize=14)
ax.legend(title='Energy Region', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(data_dir / 'janis_region_comparison.png', dpi=150)
plt.show()

## 7. Statistical Summary Across Libraries

Let's create a comprehensive statistical comparison.

In [ ]:
# Calculate statistics
stats_data = []
reference = library_data['ENDF/B-VIII.0']

for lib in libraries:
    data = library_data[lib]
    
    if lib != 'ENDF/B-VIII.0':
        ratio = data['xs'] / reference['xs']
        mean_ratio = np.mean(ratio)
        std_ratio = np.std(ratio)
        max_dev = np.max(np.abs(ratio - 1.0)) * 100  # Max deviation in %
    else:
        mean_ratio = 1.0
        std_ratio = 0.0
        max_dev = 0.0
    
    stats_data.append({
        'Library': lib,
        'Mean_Ratio': mean_ratio,
        'Std_Dev_Ratio': std_ratio,
        'Max_Deviation_%': max_dev
    })

df_stats = pd.DataFrame(stats_data)

print("\nStatistical Comparison of Libraries (relative to ENDF/B-VIII.0):")
print("="*70)
print(df_stats.to_string(index=False))

# Save to CSV
csv_file = data_dir / 'janis_library_statistics.csv'
df_stats.to_csv(csv_file, index=False)
print(f"\nStatistics saved to: {csv_file}")

## 8. Creating a Combined Dataset for ML

Let's create a comprehensive dataset with all libraries for machine learning applications.

In [ ]:
# Create combined DataFrame
# Use a subset of energy points for manageable size
energy_grid = np.logspace(-2, 7, 1000)

# Interpolate all libraries to the same energy grid
from scipy.interpolate import interp1d

combined_data = {'Energy_eV': energy_grid}

for lib in libraries:
    data = library_data[lib]
    # Create interpolation function
    f_interp = interp1d(data['energy'], data['xs'], 
                        kind='linear', fill_value='extrapolate')
    # Interpolate to common grid
    combined_data[lib.replace('/', '_').replace('-', '_').replace('.', '_')] = f_interp(energy_grid)

df_combined = pd.DataFrame(combined_data)

# Add derived features
df_combined['Log10_Energy'] = np.log10(df_combined['Energy_eV'])

# Calculate statistics across libraries
lib_columns = [col for col in df_combined.columns if col not in ['Energy_eV', 'Log10_Energy']]
df_combined['Mean_XS'] = df_combined[lib_columns].mean(axis=1)
df_combined['Std_XS'] = df_combined[lib_columns].std(axis=1)
df_combined['Min_XS'] = df_combined[lib_columns].min(axis=1)
df_combined['Max_XS'] = df_combined[lib_columns].max(axis=1)
df_combined['Range_XS'] = df_combined['Max_XS'] - df_combined['Min_XS']
df_combined['CoV_%'] = (df_combined['Std_XS'] / df_combined['Mean_XS']) * 100  # Coefficient of variation

# Save to CSV
csv_file = data_dir / 'janis_combined_libraries.csv'
df_combined.to_csv(csv_file, index=False)

print(f"Combined dataset saved to: {csv_file}")
print(f"\nDataset shape: {df_combined.shape}")
print(f"Columns: {list(df_combined.columns)}")
print("\nFirst few rows:")
print(df_combined.head())

## 9. Uncertainty Quantification from Library Spread

The spread between different libraries can be used as an estimate of evaluation uncertainty.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Top panel: Mean ± std across libraries
ax1.loglog(df_combined['Energy_eV'], df_combined['Mean_XS'], 
           'k-', linewidth=2, label='Mean across libraries')
ax1.fill_between(df_combined['Energy_eV'], 
                 df_combined['Mean_XS'] - df_combined['Std_XS'],
                 df_combined['Mean_XS'] + df_combined['Std_XS'],
                 alpha=0.3, color='blue', label='±1σ (library spread)')
ax1.fill_between(df_combined['Energy_eV'],
                 df_combined['Min_XS'],
                 df_combined['Max_XS'],
                 alpha=0.1, color='red', label='Min-Max range')

ax1.set_ylabel('Fission Cross-section (barns)', fontsize=14)
ax1.set_title('U-235 Fission: Uncertainty from Library Spread', fontsize=16)
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3, which='both')
ax1.set_xlim(1e-2, 1e7)

# Bottom panel: Coefficient of variation
ax2.semilogx(df_combined['Energy_eV'], df_combined['CoV_%'], 
             'purple', linewidth=2)
ax2.set_xlabel('Neutron Energy (eV)', fontsize=14)
ax2.set_ylabel('Coefficient of Variation (%)', fontsize=12)
ax2.set_title('Relative Spread Between Libraries', fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1e-2, 1e7)

plt.tight_layout()
plt.savefig(data_dir / 'janis_uncertainty_analysis.png', dpi=150)
plt.show()

print(f"\nMean coefficient of variation: {df_combined['CoV_%'].mean():.3f}%")
print(f"Max coefficient of variation: {df_combined['CoV_%'].max():.3f}%")
print(f"Energy with max disagreement: {df_combined.loc[df_combined['CoV_%'].idxmax(), 'Energy_eV']:.2e} eV")

## 10. Real JANIS Data Access

### Method 1: Manual Download via Web Interface

1. Visit https://www.oecd-nea.org/janisweb/
2. Navigate to "Nuclear Data" → "Cross Sections"
3. Select:
   - Isotope: e.g., "92-U-235"
   - Reaction: e.g., "(n,f)"
   - Libraries: Check boxes for ENDF, JEFF, JENDL, etc.
4. Click "Plot"
5. Use "Export" button to download as CSV
6. Load in Python:

```python
df = pd.read_csv('janis_export.csv')
```

### Method 2: Automated Web Scraping (Advanced)

```python
# This is a template - actual implementation depends on JANIS website structure
import requests
from bs4 import BeautifulSoup

session = requests.Session()
# Send POST request with query parameters
# Parse response HTML
# Extract data tables
```

### Method 3: Alternative - Direct Library Access

Instead of JANIS, download from each library directly:
- ENDF: https://www.nndc.bnl.gov/endf/
- JEFF: https://www.oecd-nea.org/dbdata/jeff/
- JENDL: https://wwwndc.jaea.go.jp/jendl/

Then process with OpenMC as in Tutorial 1.

## 11. ML Applications with Multi-Library Data

Having multiple evaluations enables powerful ML applications:

### 1. Ensemble Methods
- Use multiple libraries as an ensemble
- Average predictions weighted by library quality

### 2. Uncertainty Quantification
- Library spread → evaluation uncertainty
- Train models to predict uncertainty

### 3. Outlier Detection
- Identify libraries that disagree significantly
- Flag energy regions needing re-evaluation

### 4. Interpolation
- Use all libraries to constrain interpolation
- Predict cross-sections at unmeasured energies

### 5. Systematic Error Detection
- Compare library-specific biases
- Identify evaluation methodology issues

In [ ]:
# Example: Simple anomaly detection
# Find energies where libraries disagree most

threshold = 1.0  # 1% coefficient of variation
high_uncertainty = df_combined[df_combined['CoV_%'] > threshold]

print(f"\nEnergy points with > {threshold}% library disagreement: {len(high_uncertainty)}")
print("\nTop 10 energies with highest disagreement:")
print(high_uncertainty.nlargest(10, 'CoV_%')[['Energy_eV', 'Mean_XS', 'CoV_%']])

# Visualize
plt.figure(figsize=(14, 6))
plt.scatter(df_combined['Energy_eV'], df_combined['CoV_%'], 
            c=df_combined['CoV_%'], cmap='Reds', s=20, alpha=0.6)
plt.colorbar(label='CoV (%)')
plt.axhline(threshold, color='blue', linestyle='--', linewidth=2, 
            label=f'Threshold ({threshold}%)')
plt.xscale('log')
plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('Coefficient of Variation (%)', fontsize=14)
plt.title('Identifying High-Uncertainty Energy Regions', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Key Takeaways

1. **JANIS provides unified access** to multiple nuclear data libraries
2. **Multi-library comparison** reveals evaluation uncertainties
3. **Library spread** can be used for uncertainty quantification
4. **Different libraries** may disagree in specific energy regions
5. **Automated comparison** enables systematic validation studies

### Best Practices:
- Always compare multiple libraries for critical applications
- Use library spread as a measure of evaluation uncertainty
- Investigate regions of high disagreement
- Consider experimental data (EXFOR) for validation
- Document which library you used for calculations

## Next Steps

- Access real JANIS data for your isotopes of interest
- Compare all major libraries (ENDF, JEFF, JENDL, TENDL)
- Use multi-library data for ML ensemble methods
- Investigate discrepancies with EXFOR experimental data
- Create automated validation pipelines

## Resources

- JANIS Web: https://www.oecd-nea.org/janisweb/
- OECD NEA: https://www.oecd-nea.org/
- Nuclear Data Evaluation: https://www-nds.iaea.org/
- OpenMC (for processing files): https://docs.openmc.org/